# 01 · Finding and downloading real data

In notebook 00 we *made* a light curve. Now we'll *download* one that a real space telescope recorded — and learn the vocabulary of how astronomical data is organized and stored.

**You'll learn:** which missions produced this data · what the **MAST archive** is · the jargon (cadence, quarters, sectors, data products) · how to search and download with **`lightkurve`** · what a `LightCurve` object contains.

## Where light curves come from

A handful of space telescopes stared at the same patches of sky for a long time specifically to catch transits:

| Mission | Years | What it did |
|---|---|---|
| **Kepler** | 2009–2013 | Stared at one 115-square-degree field in Cygnus for 4 years. Found most known exoplanets. Data is split into ~3-month **quarters** (Q0–Q17). |
| **K2** | 2014–2018 | Kepler's second life after a hardware failure; observed along the ecliptic in **campaigns**. |
| **TESS** | 2018– | Surveys almost the *whole sky* in 27-day **sectors**, focusing on bright, nearby stars. Still running. |

All of this data is public and lives at **MAST** (the Mikulski Archive for Space Telescopes), hosted by STScI. The `lightkurve` library is a friendly Python front-end to MAST.

**Jargon that matters:**
- **Cadence** — how often a brightness measurement was taken. Kepler *long cadence* = every 29.4 min; *short cadence* = every ~1 min.
- **Quarter / sector / campaign** — a chunk of continuous observing. Data comes split this way, so a full light curve is usually assembled from several pieces.
- **Data product** — one downloadable file (e.g. one star, one quarter).

📖 *Resources:* [MAST archive](https://archive.stsci.edu/) · [Kepler](https://en.wikipedia.org/wiki/Kepler_space_telescope) · [TESS](https://en.wikipedia.org/wiki/Transiting_Exoplanet_Survey_Satellite)

## Searching MAST with lightkurve

`search_lightcurve()` queries MAST but downloads nothing yet — it just returns a table of *what's available*. Let's search for **Kepler-8** (catalog id `KIC 6922244`), a star we'll use throughout.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import lightkurve as lk

search = lk.search_lightcurve('KIC 6922244', author='Kepler', cadence='long')
print(f'{len(search)} data products available (one per quarter)')
search

Each row is one quarter. The `author='Kepler'` filter asks for the official mission-produced light curves (other pipelines also publish to MAST). Now let's actually **download one quarter** — `.download()` fetches a single product and returns a `LightCurve` object.

In [ ]:
lc = search[3].download()   # grab one quarter
lc.scatter(s=1)
plt.show();

## Anatomy of a `LightCurve` object

That object is essentially a table with astronomy conveniences bolted on. The two columns that matter most:

- `lc.time` — timestamps (in days; Kepler uses a system called BKJD)
- `lc.flux` — brightness at each time (raw units are electrons/second)

It also carries metadata (`lc.meta`) like the star's ID and the quarter number.

In [ ]:
print('number of measurements :', len(lc))
print('time span (days)        :', round((lc.time.max() - lc.time.min()).value, 1))
print('flux units              :', lc.flux.unit)
print('quarter                 :', lc.meta.get('QUARTER'))
print('target                  :', lc.meta.get('OBJECT'))

## Normalizing and combining quarters

Raw flux is in electrons/second and drifts between quarters. As in notebook 00, we usually **normalize** to ~1.0. And because one quarter is only ~90 days, we typically download *several* and stitch them into one long curve — more transits stacked means a stronger signal.

`download_all()` fetches every product in a search; `stitch()` normalizes each and joins them.

In [ ]:
lcc = search[1:5].download_all()      # four quarters
long_lc = lcc.stitch()                 # normalize each + concatenate
print('stitched length:', len(long_lc), 'points over',
      round((long_lc.time.max() - long_lc.time.min()).value, 0), 'days')
long_lc.scatter(s=1)
plt.show();

## The same pipeline, packaged

Those four steps — search, download every quarter, normalize-and-stitch, drop the gaps —
are the opening of every notebook that follows, so they live in `skyplay.data` as
`load_stitched`. It adds two things worth having:

- **A cache.** The stitched result is written to `data/cache/` as Parquet, so the second
  run is instant instead of re-downloading ~14,000 cadences.
- **Quarters by number.** Above we wrote `search[1:5]`, which relies on the archive
  returning products in quarter order. It does today, but that's a property of the
  response, not a guarantee. `load_stitched(quarters=(1, 2, 3, 4))` says what it means.

`skyplay.data.TARGETS` also carries published values for the stars used in this repo, so
later notebooks can check their answers instead of trusting them.

In [ ]:
from skyplay import data

target = data.TARGETS['kepler-8']
print(target, '->', target.note)
print(f'published period {target.period} d, Rp/Rs {target.rp_rs}')

# One line for everything above.
packaged = data.load_stitched('kepler-8')          # quarters 1-4, cached after this
by_hand = long_lc.remove_nans()

print(f'\nby hand  : {len(by_hand)} cadences')
print(f'packaged : {len(packaged)} cadences')
print('identical:', np.allclose(by_hand.flux.value, packaged.flux.value))

Notice the slow wobbles — that's stellar variability and instrument drift, *not* planets. Removing that safely is notebook 03's job.

## Recap
- Real light curves come from **Kepler / K2 / TESS**, archived at **MAST**.
- Data is chunked into **quarters/sectors**; `lightkurve` searches and downloads it.
- A `LightCurve` is a `time` + `flux` table; we **normalize** to ~1.0 and **stitch** quarters together.

## Learning resources
- 📗 [Lightkurve: searching & downloading data](https://docs.lightkurve.org/tutorials/1-getting-started/searching-for-data-products.html)
- 🏛️ [MAST archive portal](https://archive.stsci.edu/)
- 🚀 [Kepler mission (NASA)](https://science.nasa.gov/mission/kepler/)
- 🌍 [Kepler space telescope (Wikipedia)](https://en.wikipedia.org/wiki/Kepler_space_telescope)

**Next:** `02_from_pixels_to_light_curve.ipynb` — where does that single flux number actually come from?